In [ ]:
# Restore this study's versioned data. No API key or paid service is used.
from pathlib import Path
import sys
STUDIES = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "common/study_data.py").is_file()), None)
if STUDIES is None:
    raise RuntimeError("Open this notebook from its case-study folder in the cookbook checkout.")
sys.path.insert(0, str(STUDIES / "common"))
from study_data import ensure_data
ensure_data("ask-before-acting", root=STUDIES)


# Ask Before Acting: multi-turn GRPO against a simulated user

A LoRA adapter on GLM 5.3 Flash that asks one question when a request is missing a detail and acts without asking when it is not, trained by reinforcement learning against a simulated user.

Served through the serverless chat API, the adapter solves 62.5% of 16 held-out tasks in all four attempts against 50.0% for the base model, a paired difference of +12.5 points with a 95% interval of [-19, +44]; the figure in [section 1](#1.-See-the-result-first) shows both arms. Measured in-session during training, the same adapter went from 87.5% to 100%.

**Needs:** Python 3.12; from `cookbook/training` run `uv venv --python 3.12 && source .venv/bin/activate && uv pip install -e .`; open Jupyter from this folder (`rl_ask_before_acting/`, the first cell checks); `../common/figures.py` is imported through `aba_figures.py`.

**Status:** Recorded results are read from committed files; Run All makes no API call and spends nothing. Two paid knobs, both off by default and both needing `FIREWORKS_API_KEY` in the environment:

- `RUN_TRAINING` (section 4): a two-step smoke run on serverless training, 2 steps of 4 tasks x 4 episodes, costs cents; writes to `runs/smoke/`, which is gitignored.
- `RUN_SERVED_EVAL` + `ADAPTER_MODEL` (section 5): 16 tasks x 4 attempts x 2 arms = 128 episodes through the chat API, each 2 to 5 short calls; cost not measured, expect cents.

**Three routes:** Explore the result (sections 1 to 3, offline) · Reproduce it (section 4 smoke run and section 5 served evaluation, paid) · Use your own data (section 6).

Outputs headed `RECORDED RESULT` were read from `results/`. Cells headed `LIVE RUN` do nothing until you flip a flag, and a recorded success never means your run worked.


**Glossary**

- **LoRA / adapter / rank**: a small set of extra weights trained on top of a frozen base model (here rank 64, the width of those extra matrices); "adapter" means that trained set.
- **Base vs adapter arm**: the untouched GLM 5.3 Flash vs the same model with the adapter loaded; both see the same system prompt. There is no separate prompted arm here, since the prompt already says to ask.
- **GRPO**: Group Relative Policy Optimization, the RL method used: sample a group of episodes per task, score them, and push the model toward the above-average ones.
- **Group / advantage / datum**: a group is the 8 episodes sampled for one task; the advantage is each episode's reward standardised within its group; a datum is one training example (here one assistant turn with its advantage).
- **Episode / task**: a task is one request with a hidden spec; an episode is one attempt at it, ending in an action tool call, a prose reply, or the turn limit.
- **Held-out**: 16 tasks never trained on, used only for evaluation.
- **Pass rate / pass^4**: pass rate is the share of attempts that were correct; pass^4 is the share of tasks correct in all four of their attempts.
- **Wilson interval**: a 95% confidence interval for a proportion that behaves sensibly at small counts (16 tasks).
- **Paired delta / cluster bootstrap**: the adapter-minus-base difference on the same tasks, with an interval from resampling tasks (not attempts, since four attempts share a task).
- **Temperature 1**: sampling with the model's full randomness, so repeated attempts can differ; greedy decoding would always pick the same answer.
- **Sampler / renderer**: the sampler is the serverless service that generates text from the current adapter during training; the renderer turns messages and tool calls into tokens and back.
- **Importance-sampling loss / Adam**: the GRPO loss weights each sampled token by its advantage and by how the current model's probability compares with the sampling model's; Adam is the optimizer that applies the gradient.
- **Checkpoint vs promoted model**: a checkpoint is adapter weights saved during a run; promoting one registers it as a model you can load for inference.
- **Serverless training vs serverless inference**: training bills per token on a shared pool with nothing to provision; serverless inference serves the base model (and a loaded adapter) through the public chat API, also per token. A dedicated deployment is reserved GPUs instead.
- **`reasoning_effort`**: a chat API parameter that limits how much the model thinks before answering; the served evaluation used `low`.
- **`firectl`**: the Fireworks command-line tool, used here only to load a promoted adapter onto serverless inference.


## 1. See the result first

A task is a request like "Book a meeting the budget sync on Monday starting 16:00 not too long." One
episode is one attempt at it: the agent may ask the simulated user questions, then calls an action tool,
and the episode is correct when the tool and every argument match the hidden spec (here, 30 minutes).
Each of the 16 held-out tasks was attempted four times at temperature 1, so a task can be correct in
zero to four of its attempts; "all four correct" is the strict version, since one guessed value in four
tries is a failure a user would meet. The share of tasks correct in all four attempts is written
**pass^4** below, and it is the headline because greedy decoding already reads 100% before training.

Two evaluations are committed, and they are not the same conditions. **Served** (2026-10-12, first) is
the one a user would see: the promoted adapter loaded onto serverless inference, base model and adapter
both called through the public chat API with `reasoning_effort=low`. **In-session** (2026-10-09, second)
is the training-time view: the step-0 and step-20 checkpoints sampled through the training session's own
sampler. Whiskers are 95% intervals: Wilson score on the 16 per-task all-correct flags for pass^4, and a
bootstrap that resamples whole tasks (not attempts, since four attempts share a task) for the attempt
rate.

In [ ]:
import json, os, sys
from pathlib import Path

from aba_figures import IN_SESSION, SERVERLESS, load_arms, load_jsonl, plot_result, print_result, recorded  # stats + plots; draws with ../common/figures.py

HERE = Path.cwd()
assert (HERE / "aba_env.py").exists(), "Open the notebook from the rl_ask_before_acting folder."

ARMS = load_arms()          # {"serverless" | "in_session": {"Base": episodes, "Adapter": episodes}}
plot_result(ARMS)
print_result(ARMS)


**What to see:** the adapter bar is above the base bar in both evaluations, but on the served side the whiskers overlap almost entirely.

Served, the margin is small: 62.5% against 50.0% of tasks in all four attempts and 89.1% against 84.4%
of attempts, two tasks and three attempts out of 16 and 64. In-session the step-20 checkpoint is
correct in every one of its 64 attempts, up from 87.5% of tasks and 95.3% of attempts at step 0, and
even that 100% carries a Wilson interval down to 81%. Per-arm whiskers do not answer "is the adapter
better"; the paired difference below does.


The honest size of the gain is the paired difference per task, since both arms answered the same 16
requests. One bar per evaluation: adapter all-four-correct minus base all-four-correct, bootstrapped over
the 16 task pairs with `paired_delta_ci`.

In [ ]:
from aba_figures import plot_paired_delta

plot_paired_delta(ARMS)


**What to see:** both bars sit at +12.5 points, and the served whisker crosses zero.

Both deltas are +12.5 points (two tasks out of sixteen; 95% paired bootstrap over the 16 task pairs).
Served, the interval runs from -19 to +44 points, so the data allows anything from a small loss to a
large gain; in-session it runs from 0 to +31 because the adapter had no misses to pair against.

One failure mode the reward charges for is answering in prose instead of calling a tool; the chart below
shows how often each arm did that. The other base-model failures are guessed values and paraphrased
message bodies, which section 2 shows on a transcript.


In [ ]:
from aba_figures import plot_prose_rate

plot_prose_rate(ARMS)


**What to see:** the adapter bars are at zero in both evaluations; the base bars are small, with whiskers that reach zero.

The adapter never answered in prose in either evaluation (0 of 64 twice), against 2 and 1 base attempts;
whiskers are a task-level bootstrap, and with counts this small they say only that the base rate is
under about 8%, so read this as "a base failure mode not seen in the adapter", not as a measured reduction.

The in-session run also evaluated at steps 5, 10 and 15, so the trajectory of the headline is on record.


In [ ]:
from aba_figures import plot_training_curve

evals = plot_training_curve()


**What to see:** pass^4 climbs in two single-task steps (at step 5 and at step 20) while attempt accuracy barely moves.

Point estimates from the in-session evaluations, no interval. One task was fixed by step 5 and the
second between steps 15 and 20, with questions per task pinned at 1.00 and unneeded questions at 0.00
throughout: the asking behaviour never changed, only the consistency of the final action, and a one-task
step is inside the noise of any single 16-task evaluation.


## 2. One episode, step by step (offline)

The numbers hide what a good episode looks like. This section runs the coded environment locally: a
seeded task, a scripted trajectory, the simulated user's replies from the hidden spec, and the scorer's
verdict. No model is involved; the assistant turns are written out by hand so the reward is legible.
In `play` below, `revealed` is the set of fields the simulated user has already answered, which is how
it knows to say "I already told you that" the second time.


In [ ]:
from aba_env import (SYSTEM_PROMPT, generate_tasks, fields_asked_about, score_episode,
                     simulated_user_reply, tool_specs)


def play(task, assistant_turns):
    '''Run scripted assistant turns through the simulator, the same way ask_before_acting_rl.run_episode does.'''
    print("USER:     ", task["request"])
    revealed, unnecessary, questions, action_call = set(), 0, 0, None
    for call in assistant_turns:
        if call["name"] == "ask_user":
            questions += 1
            reply, newly, unneeded = simulated_user_reply(call["arguments"]["question"], task, revealed)
            revealed.update(newly)
            unnecessary += int(unneeded)
            print("ASSISTANT: ask_user(question=%r)" % call["arguments"]["question"])
            print("USER:     ", reply, "   <- unneeded question" if unneeded else "")
            continue
        print(f"ASSISTANT: {call['name']}({', '.join(f'{k}={v!r}' for k, v in call['arguments'].items())})")
        action_call = call
        break
    score = score_episode(task, action_call, unnecessary, questions)
    print(f"SCORE:     reward={score['reward']:.2f}  right_tool={score['right_tool']}  right_args={score['right_args']}  "
          f"questions={score['questions']}  unnecessary={score['unnecessary_questions']}\n")
    return score


task = generate_tasks(1, seed=22)[0]
print("hidden spec:", task["spec"])
print("missing or vague fields the agent must ask about:", task["missing"], "\n")

good = play(task, [
    {"name": "ask_user", "arguments": {"question": "How long should the budget sync be?"}},
    {"name": "book_meeting", "arguments": {"title": "budget sync", "day": "Monday", "start_time": "16:00",
                                           "duration_minutes": 30}},
])

bad = play(task, [
    {"name": "book_meeting", "arguments": {"title": "budget sync", "day": "Monday", "start_time": "16:00",
                                           "duration_minutes": 60}},
])

"Not too long" reads like information but the hidden spec says 30, so the guess of 60 scores zero, and
nothing it got right counts. The good trajectory asks about exactly the open field, gets "30 minutes.",
books, and scores 1.0. The second example is the opposite situation: a request with nothing missing,
where asking anything is the mistake.

In [ ]:
full = next(t for t in generate_tasks(200, seed=0) if not t["missing"])
print("hidden spec:", full["spec"], "   missing:", full["missing"], "\n")

act_now = play(full, [
    {"name": "set_reminder", "arguments": {"text": "water the plants", "day": "Monday", "time": "08:30"}},
])

asked_anyway = play(full, [
    {"name": "ask_user", "arguments": {"question": "Which day should I set it for?"}},
    {"name": "set_reminder", "arguments": {"text": "water the plants", "day": "Monday", "time": "08:30"}},
])

The simulated user says "I already told you that", the question is charged as unnecessary, and the
reward falls from 1.0 to 0.75 even though the action was right. A question about an open field is free
only the first time: each further question costs 0.1, which is what pushes the policy to bundle
everything it needs into one.

The recorded serverless evaluation has the real thing. Below is the adapter guessing a vague time on one
held-out task, straight from `episodes.jsonl`.

In [ ]:
def show_recorded(ep):
    print("USER:     ", ep["task"]["request"], "   | hidden:", ep["task"]["spec"], "| must ask:", ep["task"]["missing"])
    for m in ep["transcript"][1:]:
        if m["role"] == "assistant":
            for tc in m.get("tool_calls") or []:
                fn = tc["function"]
                print(f"ASSISTANT: {fn['name']}({fn['arguments']})")
            if m.get("content"):
                print("ASSISTANT:", m["content"])
        elif m["role"] == "tool":
            print("USER:     ", m["content"])
    s = ep["score"]
    print(f"SCORE:     reward={s['reward']:.2f}  right_args={s['right_args']}  ended={ep['ended']}\n")


recorded(f"{SERVERLESS}/episodes.jsonl, adapter arm")
miss = next(e for e in ARMS["serverless"]["Adapter"] if not e["score"]["right_args"] and e["task"]["vague"])
show_recorded(miss)
fixed = next(e for e in ARMS["serverless"]["Adapter"] if e["task"]["request"] == miss["task"]["request"] and e["score"]["right_args"])
show_recorded(fixed)

Same task, two of its four samples: one guesses "afternoon" as 13:00 and scores zero, the other asks and
scores 1.0. That inconsistency is the whole headroom in this environment, and pass^4 is the measure that
sees it.

## 3. The environment, explained

Everything the policy is scored against is in `aba_env.py`, pure Python with a seeded generator. Four
pieces: the tool specs the model sees, the task generator that hides fields, the simulated user that
answers from the hidden spec, and the scorer. Each is a function you can call.

In [ ]:
from IPython.display import Markdown, display

rows = ["| Tool | Arguments | Description |", "|---|---|---|"]
for spec in tool_specs():
    args = ", ".join(f"`{k}` ({v['type']})" for k, v in spec["parameters"]["properties"].items())
    rows.append(f"| `{spec['name']}` | {args} | {spec['description']} |")
display(Markdown("\n".join(rows)))
print("System prompt:\n ", SYSTEM_PROMPT)

**The generator.** Each task draws a full spec, omits zero, one or two fields outright, and phrases each
remaining field vaguely with probability 0.35. A vague field joins the omitted ones in `missing`, because
to the scorer it is equally unknown. The committed task file holds 600 of these; the held-out 16 are
drawn from it by a fixed seed.

In [ ]:
from collections import Counter

TASK_FILE = Path("data/aba_tasks.jsonl")   # gitignored; `python ask_before_acting_rl.py --prepare-dataset` writes it
if TASK_FILE.exists():
    tasks = [json.loads(r["ground_truth"]) for r in load_jsonl(TASK_FILE)]
else:
    from aba_env import generate_tasks
    tasks = generate_tasks(600, seed=0)   # the same seeded set the training script writes
n = len(tasks)
print(f"{n} tasks from", "data/aba_tasks.jsonl" if TASK_FILE.exists() else "generate_tasks(600, seed=0) (task file not written yet)")
print("unknown fields per task:", dict(sorted(Counter(len(t['missing']) for t in tasks).items())))
print(f"fully specified (asking anything is a mistake): {sum(not t['missing'] for t in tasks) / n:.1%}")
print(f"contain at least one vague detail:               {sum(bool(t['vague']) for t in tasks) / n:.1%}")
print("actions:", dict(Counter(t["action"] for t in tasks)))
print()
for t in generate_tasks(4, seed=3):
    print(f"{t['request']:<70} omitted={[f for f in t['missing'] if f not in t['vague']]} vague={t['vague']}")

**The simulated user** is a keyword matcher, not a model. `fields_asked_about` maps a question to fields
by cue words, after blanking out values the request already stated so that echoing the request ("remind
you to call the dentist") is not read as asking about it. `simulated_user_reply` then answers every
needed field the question names, says "I already told you that" when any named field was already given,
and "Whatever you think is best" when it cannot map the question to anything, which costs a turn but is
not charged as unnecessary. The last case is the environment's known limit.

The cell below sends five questions through the matcher. What to take from it: the first four show the
four outcomes (answered, partly answered, charged as already told, unmappable); the fifth is a real miss
from the served evaluation (held-out task 14) where the matcher could not tell what the question was asking.


In [ ]:
t = generate_tasks(1, seed=22)[0]       # "Book a meeting the budget sync on Monday starting 16:00 not too long."
print(t["request"], "| missing:", t["missing"])
for q in ["How long should the budget sync be?",
          "When should the meeting be and how long?",          # 'when' is ambiguous; day was stated, so only the open field is answered
          "Which day is the budget sync?",                      # already stated: charged
          "Any other details?",                                 # unmappable: not charged, no answer
          "What should the Friday 13:00 reminder be about?"]:   # a real matcher miss from the serverless eval (task 14)
    task_for_q = t if "reminder" not in q else next(e["task"] for e in ARMS["serverless"]["Base"] if e["task_index"] == 14)
    fields = fields_asked_about(q, task_for_q["action"], task_for_q)
    reply, newly, unneeded = simulated_user_reply(q, task_for_q, set())
    print(f"  Q: {q:<48} fields={fields!s:<28} -> {reply!r:<30} unneeded={unneeded}")

**The scorer** normalises arguments before comparing, so `2pm` equals `14:00`, `text` equals `sms` and
`thursday` equals `Thursday`; anything else that differs from the hidden spec scores zero.

In [ ]:
spec = t["spec"]
variants = {
    "exact":            dict(spec),
    "2pm-style time":   {**spec, "start_time": "4pm"},
    "string duration":  {**spec, "duration_minutes": "30 minutes"},
    "wrong duration":   {**spec, "duration_minutes": 45},
    "missing argument": {k: v for k, v in spec.items() if k != "title"},
    "wrong tool":       None,
}
for name, args in variants.items():
    call = {"name": "set_reminder", "arguments": dict(spec)} if args is None else {"name": "book_meeting", "arguments": args}
    s = score_episode(t, call, unnecessary_questions=0, questions=1)
    print(f"{name:<18} reward={s['reward']:.2f}  right_tool={s['right_tool']!s:<5} right_args={s['right_args']}")
print()
print("reward with 1 question, 0 unneeded:", score_episode(t, {"name": "book_meeting", "arguments": spec}, 0, 1)["reward"])
print("reward with 3 questions, 1 unneeded:", score_episode(t, {"name": "book_meeting", "arguments": spec}, 1, 3)["reward"])

## 4. The multi-turn GRPO loop

Terms used below: **GRPO** samples a **group** of episodes per task and scores them; each episode's
**advantage** is its reward standardised within the group; a **datum** is one training example; the
**sampler** is the serverless service that generates text from the current adapter; the **renderer**
turns messages and tool calls into tokens and back.

`ask_before_acting_rl.py` is one file and one loop, so the phases are easier to read in prose than in
code. What to take from the list: steps 1 to 3 produce scored episodes, step 4 decides which of them
can teach anything, and steps 5 and 6 are the actual update and the check. Each optimizer step:

1. **Snapshot** the current LoRA weights for the sampler (`save_weights_for_sampler`).
2. **Roll out a group.** Take 8 training tasks; for each, run `group_size` = 8 episodes at temperature 1
   through the serverless sampler. An episode is the loop from section 2: the policy samples a turn, the
   renderer parses it into a tool call, `ask_user` goes to `simulated_user_reply` and the answer is
   appended as a tool message, an action tool ends the episode, and so does prose or the 4-turn limit.
3. **Score** every episode with `score_episode`.
4. **Normalise advantages across the group.** Rewards within one task's 8 episodes are standardised
   (mean 0, unit variance). A group whose rewards are all equal has nothing to teach and is dropped;
   `train/kept_groups` in `metrics.jsonl` counts the survivors.
5. **One optimizer step.** Every assistant turn of every kept episode becomes a datum for the
   importance-sampling loss, then one Adam step at 2e-5.
6. **Checkpoint and evaluate** every 5 steps on the 16 held-out tasks, and at the end.

**The one thing that differs from single-turn GRPO.** A trajectory here has tool and user turns in it,
and the policy did not write those. So the episode is split into one datum per assistant turn: the datum's
input is the whole conversation up to that turn (system prompt, tools, request, earlier questions and
the simulated user's replies), its advantage is zero on every prompt token and equal to the episode's
group-relative advantage on the sampled tokens only. A question that led to a correct action is
reinforced along with the action; the user's answer is context, never a target.

The advantage function below is a verbatim copy of the script's `_group_relative_advantages`, kept here
rather than imported because importing `ask_before_acting_rl.py` requires the training SDK.


In [ ]:
import math


def group_relative_advantages(rewards, eps=1e-8):
    '''Verbatim copy of ask_before_acting_rl._group_relative_advantages: standardise within one task's group.'''
    if len(rewards) <= 1:
        return [0.0 for _ in rewards]
    mean = sum(rewards) / len(rewards)
    std = math.sqrt(sum((r - mean) ** 2 for r in rewards) / (len(rewards) - 1))
    return [0.0 for _ in rewards] if std < eps else [(r - mean) / (std + eps) for r in rewards]


groups = {
    "mixed group (learns)":        [1.0, 1.0, 0.0, 1.0, 0.75, 1.0, 0.0, 1.0],
    "all solved (dropped)":        [1.0] * 8,
    "all failed the same (dropped)": [0.0] * 8,
}
for name, rewards in groups.items():
    adv = group_relative_advantages(rewards)
    kept = any(a != 0.0 for a in adv)
    print(f"{name:<32} rewards={rewards}\n{'':<32} advantages={[round(a, 2) for a in adv]}  kept={kept}")


That drop rule is why training signal on a near-saturated environment is thin: the base model already
solves most groups every time. The recorded run logs how many of the 8 groups survived at each step.

In [ ]:
from aba_figures import plot_kept_groups

train = plot_kept_groups()


**What to see:** the orange part of each bar is never more than half of it, and several steps have none.

Between 0 and 4 of 8 groups per step carried signal, and a step with zero kept groups is a step with no
gradient; a harder environment, or a higher temperature, is how you would buy more of them.

### LIVE RUN: a two-step smoke run (off by default)

Nothing below runs unless you flip `RUN_TRAINING` to `True` and `FIREWORKS_API_KEY` is set. This calls the script's
`main` with the smoke arguments from the detailed guide (`GUIDE.md`): 2 steps of 4 tasks x 4 episodes, which costs cents on
serverless training and regularly keeps zero groups, so it proves the mechanics, not the result. The
full run is `runs/run_aba_glm53_flash.sh` (20 steps, 8 x 8). Output lands in `runs/smoke/`, which is
gitignored; the recorded results above do not change. Left `False`, the cell prints one line and exits.


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    assert os.environ.get("FIREWORKS_API_KEY"), "Supply your key through the environment."
    sys.path.insert(0, str(HERE.parents[2]))        # the cookbook root, for training.renderer and friends
    from ask_before_acting_rl import main as train_main
    train_main([
        "--dataset", str(HERE / "data/aba_tasks.jsonl"),
        "--steps", "2", "--prompt-groups-per-step", "4", "--group-size", "4",
        "--eval-prompt-groups", "4", "--eval-group-size", "4",
        "--checkpoint-name", "aba-smoke", "--final-checkpoint-name", "aba-smoke-final",
        "--run-dir", str(HERE / "runs/smoke"),
    ])
else:
    print("LIVE RUN (disabled): serverless training off. Flip RUN_TRAINING to spend a few cents on a smoke run.")

## 5. Evaluate a served adapter (LIVE RUN, off by default)

The in-session evaluation samples through the training session; the number a user sees comes from the
inference API. `evaluate_served.py` runs the same 16 held-out tasks (read from a stored transcript file so
the task set is exactly the run's), the same system prompt, tool schemas, simulated user and scorer
against any served model through the chat completions API. That is how the served bars in the section 1
figure were made, and both arms scored lower there than through the session sampler (base 50.0% against 87.5%
pass^4), so the two evaluations are not the same conditions and should not be read against each other.

**The knob.** `RUN_SERVED_EVAL` is `False` by default and the cell then only prints where the recorded
evaluation lives. Set to `True`, it needs `FIREWORKS_API_KEY` and makes 16 tasks x 4 attempts x 2 arms =
128 episodes of chat API calls, each 2 to 5 short calls (cost not measured; expect cents). Output lands
in `results/my-served-eval/`, which is gitignored.

**Prerequisite: an adapter the chat API can reach.** Promote the final checkpoint of a training run by
passing `--output-model-id accounts/<account>/models/<name>` to `ask_before_acting_rl.py`, then load it
onto serverless inference with the Fireworks CLI, `firectl`:

```
firectl -a <account> model load-lora <name> --serverless --wait
```

Put the resulting model id in `ADAPTER_MODEL` below, then flip the flag.


In [ ]:
RUN_SERVED_EVAL = False
ADAPTER_MODEL = "accounts/YOUR_ACCOUNT/models/YOUR_ADAPTER"

if RUN_SERVED_EVAL:
    import subprocess
    assert os.environ.get("FIREWORKS_API_KEY"), "Supply your key through the environment."
    assert "YOUR_ACCOUNT" not in ADAPTER_MODEL, "Set ADAPTER_MODEL to your loaded adapter."
    subprocess.run([
        sys.executable, str(HERE / "evaluate_served.py"),
        "--tasks-from", str(HERE / IN_SESSION / "eval_transcripts/step-0020.jsonl"),
        "--arm", "base=accounts/fireworks/models/glm-5p3-flash",
        "--arm", f"adapter={ADAPTER_MODEL}",
        "--out", str(HERE / "results/my-served-eval"),
    ], check=True)
else:
    print("LIVE RUN (disabled): chat API calls off. The recorded serverless evaluation is in", SERVERLESS)

## 6. Bring your own environment

The loop never looks inside a task; it only needs three things from `aba_env.py`, and each is a table or
one function.

1. **Actions** (`ACTIONS`): the tools the agent can end an episode with, each with typed fields. Add a
   tool with its fields and a line in `make_task` that draws a spec for it; `tool_specs()` builds the
   schemas from the table.
2. **Phrases** (`_PHRASES`, `_VAGUE`, `_FIELD_CUES`): how the request states each field exactly, how it
   states it vaguely, and which words a question would use to ask about it. The cue list is the
   simulated user's whole vocabulary, so after changing it run `fields_asked_about` on a dozen questions
   you would expect a model to write (section 3) before training; the first smoke of this environment
   blamed the model for two episodes that were really the matcher missing "first name" as a question
   about the recipient.
3. **The scorer** (`score_episode` and `_norm`): what counts as the same value. If a field has
   equivalent spellings, normalise them here, or every variant the hidden spec did not use scores zero.

Keep the shape of the reward: correctness as a gate, then small penalties for asking what you were told
and for asking more than once. If your base model already solves your environment every time at
temperature 1, there is nothing for GRPO to learn; check `rollout/correct_rate` on a smoke run before
paying for the full one, and add vague or conflicting details until it drops.

## What to take away

**Did training help?** Served through the chat API, the adapter is two tasks better than the base model
on the same 16 (62.5% against 50.0% in all four attempts, 89.1% against 84.4% of attempts), and the
paired interval on that difference runs from -19 to +44 points, so the served data allows anything from
a small loss to a large gain. In-session it went from 87.5% to 100% of tasks with questions per task
fixed at 1.00 and unneeded at 0.00, so what moved during training was consistency of the final action,
not the asking behaviour. Quote the served numbers with their interval, not the in-session 100%.

**Where did it fail?** Guessing a vague value (the adapter set a reminder for "in the afternoon" at a guessed 13:00 on one
sample), inventing a reminder text after the simulated user failed to map a question (a matcher miss,
task 14), and a request missing two fields where it asked for the day but not what the reminder was about. The base model's extra failure
mode, answering in prose, was not seen in the adapter in either evaluation (0 of 64 twice, against 2 and 1 for the base).

**Adapter, prompt, or more data?** Observations first: the system prompt already says to ask and not to
guess, the base model follows it most of the time, and the served adapter still fails about one attempt
in nine. Training signal was thin (0 to 4 of 8 groups per step) because most groups scored alike. What
this experiment can say: the adapter adds nothing to the prompt, so it is the cheaper option only if a
gain of the served size is worth a training run. What it cannot say is whether more tasks, more steps or
a harder environment would widen the gap; a harder environment is one hypothesis, suggested by the
thin signal, and the experiment below is how to test it.

**One experiment to try next.** Raise `VAGUE_PROBABILITY` from 0.35 to 0.6 and add a fourth action with
two vague fields, regenerate the task file, and run 20 steps again. If the base pass^4 drops to 60 to 70%
in-session and the adapter closes most of that gap with `kept_groups` above 4 per step, the method
scales with the headroom; if the serverless gap stays within one or two tasks, the chat API path itself
is where the next look belongs.